**Crop Disease Detection**

Deep Learning Benchmark: Custom CNN vs EfficientNetB0 vs ResNet50 vs VGG16

Dataset: PlantVillage (Apple subset) &nbsp;|&nbsp;

Framework: TensorFlow / Keras &nbsp;|&nbsp; Target: 4-class disease classification

## 1. Problem Statement

### Background

Crop diseases are responsible for 20–40% of global agricultural yield losses annually (FAO, 2021). Early and accurate identification of disease is critical but in most farming communities, this depends on expensive agronomist visits that are infrequent and slow.

### Objective

Build an **automated visual diagnosis system** that:
- Accepts a photo of an apple leaf as input
- Classifies it into one of **4 disease states**: `Apple Scab`, `Black Rot`, `Cedar Apple Rust`, or `Healthy`
- Returns a human-readable health verdict with confidence scores
- Is accurate enough for real-world deployment (target: >90%)

### Dataset

We use the **PlantVillage Dataset** (Kaggle: `tushar5harma/plant-village-dataset-updated`), which contains laboratory-quality leaf images with clean labels.

### Class	            Train	Validation	Test	Total

Apple Scab:	      2,016	453	51	2,520

Black Rot:	        1,987	447	50	2,484

Cedar Apple Rust:	1,760	396	44	2,200

Healthy:          2,008	451	51	2,510

Total:	            7,771	1,747	196	9,714


### Challenges

- **Visual similarity** between disease classes (e.g., Scab vs Black Rot can look alike)
- **Lighting and background variation** in real-world photos vs controlled lab images
- **Class imbalance**: Cedar Apple Rust has ~13% fewer samples than other classes
- **Computational constraints**: Large pretrained models (VGG16 = 138M params) are slow without a GPU


## 2. Solution Architecture

### High-Level Pipeline

```
  Raw Leaf Image (JPG/PNG)
         │
         ▼
  ┌──────────────────────┐
  │   Preprocessing       │  Resize → Normalise/preprocess_input
  └──────────────────────┘
         │
         ▼
  ┌──────────────────────┐
  │   Data Augmentation   │  Rotation, Flip, Zoom, Shift (train only)
  └──────────────────────┘
         │
     ┌───┴────────────────────────────────┐
     │              │             │       │
  Custom CNN  EfficientNetB0  ResNet50  VGG16
     │              │             │       │
     └───┬────────────────────────────────┘
         │
         ▼
  Softmax (4 classes)
         │
         ▼
  Benchmark → Best Model Selected
         │
         ▼
  Interactive Inference (upload image → verdict)
```

### Two-Phase Transfer Learning Strategy

For all three pretrained models (EfficientNetB0, ResNet50, VGG16) we use a **two-phase approach**:

**Phase 1 — Frozen Backbone (15 epochs)**
- Backbone weights are **frozen** (not updated)
- Only the custom classification head is trained
- Learning rate: `1e-3` (higher — only head params are small in count)
- Purpose: quickly adapt the head to our 4-class problem without destroying ImageNet features

**Phase 2 — Fine-tuning (10 epochs)**
- Top layers of the backbone are **unfrozen** selectively
- Learning rate: `1e-5` (very low — avoids catastrophic forgetting)
- Purpose: let the upper backbone layers specialise for leaf texture/colour patterns

---
## 3. Environment Setup

We install `kagglehub` for one-line dataset downloading. All other libraries (`tensorflow`, `sklearn`, `matplotlib`) are pre-installed in Colab.

> **GPU strongly recommended.** Enable it via *Runtime → Change runtime type → T4 GPU* before running this notebook. Training all four models takes approximately 30–60 min on a T4.

In [ ]:
!pip install -q kagglehub
import tensorflow as tf
print(f"TensorFlow : {tf.__version__}")
print(f"GPU devices: {tf.config.list_physical_devices('GPU')}")

---
## 4. Dataset Download & Exploration

`kagglehub.dataset_download()` handles authentication and caching automatically. If the dataset has already been downloaded in this Colab session it uses the cached copy, so re-running is fast.

The dataset is structured as:
```
plant-village-dataset-updated/
  Apple/
    Train/
      Apple Scab/  Black Rot/  Cedar Apple Rust/  Healthy/
    Val/   ...
    Test/  ...
  Bell Pepper/  Cherry/  Corn (Maize)/  ...
```
We focus on the **Apple** subtree for this demo but the code is parameterised to extend to all 9 plant categories.

In [ ]:
import kagglehub, os, shutil

KAGGLE_ID  = "tushar5harma/plant-village-dataset-updated"
LOCAL_ROOT = "plant-village-dataset-updated"

print(f"Downloading '{KAGGLE_ID}'...")
path = kagglehub.dataset_download(KAGGLE_ID)
os.makedirs(LOCAL_ROOT, exist_ok=True)

if os.path.isdir(path) and path != LOCAL_ROOT:
    for item in os.listdir(path):
        s, d = os.path.join(path, item), os.path.join(LOCAL_ROOT, item)
        if os.path.isdir(s):
            if os.path.exists(d): shutil.rmtree(d)
            shutil.copytree(s, d)
        else:
            shutil.copy2(s, d)

print("Done.\n")

# Explore Apple subtree
apple = os.path.join(LOCAL_ROOT, 'Apple')
print(f"{'Split':<8} {'Class':<25} {'Images':>7}")
print("-" * 45)
grand_total = 0
for split in ['Train', 'Val', 'Test']:
    sp = os.path.join(apple, split)
    if not os.path.isdir(sp): continue
    for cls in sorted(os.listdir(sp)):
        cp = os.path.join(sp, cls)
        n  = len([f for f in os.listdir(cp) if f.lower().endswith(('.jpg','.jpeg','.png'))])
        grand_total += n
        print(f"{split:<8} {cls:<25} {n:>7,}")
print("-" * 45)
print(f"{'TOTAL':<34} {grand_total:>7,}")

### Visualise Sample Images

Before training we should always visually inspect samples from each class. This catches common issues like corrupted images, wrong class assignments or extreme lighting variation.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import random, glob

CLASSES = ['Apple Scab', 'Black Rot', 'Cedar Apple Rust', 'Healthy']
random.seed(0)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
fig.suptitle('Sample Images per Disease Class (2 random samples each)', fontsize=14, fontweight='bold')

for col, cls in enumerate(CLASSES):
    imgs = glob.glob(f"{LOCAL_ROOT}/Apple/Train/{cls}/*.jpg")
    if not imgs: imgs = glob.glob(f"{LOCAL_ROOT}/Apple/Train/{cls}/*.JPG")
    samples = random.sample(imgs, min(2, len(imgs)))
    for row, img_path in enumerate(samples):
        ax = axes[row][col]
        ax.imshow(mpimg.imread(img_path))
        ax.axis('off')
        if row == 0:
            ax.set_title(cls, fontsize=11, fontweight='bold', pad=8)

plt.tight_layout()
plt.show()

---
## 5. Data Preparation & Splitting

The PlantVillage dataset already provides `Train/Val/Test` splits inside each plant folder. We simply **consolidate** these into a flat directory structure that Keras `ImageDataGenerator.flow_from_directory()` can consume.

In [ ]:
import random
random.seed(42)

SPLIT_ROOT  = 'plant_disease_dataset_split'
TRAIN_DIR   = f'{SPLIT_ROOT}/train'
VAL_DIR     = f'{SPLIT_ROOT}/validation'
TEST_DIR    = f'{SPLIT_ROOT}/test'

TARGET_CLASSES = ['Apple Scab', 'Black Rot', 'Cedar Apple Rust', 'Healthy']

if os.path.exists(SPLIT_ROOT): shutil.rmtree(SPLIT_ROOT)
for d in [TRAIN_DIR, VAL_DIR, TEST_DIR]: os.makedirs(d, exist_ok=True)

split_map = {'Train': TRAIN_DIR, 'Val': VAL_DIR, 'Test': TEST_DIR}
counts    = {}

for split_name, target_dir in split_map.items():
    src = os.path.join(LOCAL_ROOT, 'Apple', split_name)
    if not os.path.isdir(src): continue
    for cls in os.listdir(src):
        if cls not in TARGET_CLASSES: continue
        cls_src = os.path.join(src, cls)
        cls_dst = os.path.join(target_dir, cls)
        os.makedirs(cls_dst, exist_ok=True)
        imgs = [f for f in os.listdir(cls_src) if f.lower().endswith(('.jpg','.jpeg','.png'))]
        for img in imgs:
            shutil.copy(os.path.join(cls_src, img), os.path.join(cls_dst, img))
        counts.setdefault(cls, {})[split_name] = len(imgs)

print(f"{'Class':<25} {'Train':>7} {'Val':>7} {'Test':>7}")
print("-" * 50)
for cls in sorted(counts):
    tr = counts[cls].get('Train', 0)
    va = counts[cls].get('Val',   0)
    te = counts[cls].get('Test',  0)
    print(f"{cls:<25} {tr:>7,} {va:>7,} {te:>7,}")
print("Dataset ready.")

---
## 6. Shared Utilities & Config

### Global Hyperparameter Choices

| Parameter | Value | Rationale |
|-----------|------:|----------|
| `BATCH_SIZE` | 32 | Standard choice that fits in 15 GB GPU VRAM for all models; powers of 2 are efficient for GPU memory alignment |
| `EPOCHS_FROZEN` | 15 | Enough epochs for the classification head to converge without wasting compute on a frozen backbone |
| `EPOCHS_FINETUNE` | 10 | Short fine-tuning phase — the backbone is already well-initialised from ImageNet, so few updates are needed |
| `PATIENCE` | 5 | EarlyStopping patience; stops training if val_accuracy doesn't improve for 5 consecutive epochs |
| `IMG_SIZE_CUSTOM` | 150×150 | Balance between resolution and speed for the from-scratch CNN |
| `IMG_SIZE_TRANSFER` | 224×224 | The canonical input size all three pretrained models were trained on during ImageNet pretraining |

### Callbacks Used

- **`EarlyStopping`** — halts training when `val_accuracy` plateaus, saves compute and prevents overfitting
- **`ReduceLROnPlateau`** — halves the learning rate after 3 stagnant epochs; helps the model escape local minima
- **`ModelCheckpoint`** — saves the best weights so EarlyStopping can restore them even after degraded epochs

### Data Augmentation Strategy

Augmentation is applied **only to the training set**. Val/Test use raw (or only normalised) images to get unbiased metrics.

---
## 7. Model 1 — Custom CNN (Baseline)

### Architecture

```
Input (150×150×3)
  │
  ├─ Conv2D(32, 3×3, relu) → BN → MaxPool(2×2)      # 75×75×32
  ├─ Conv2D(64, 3×3, relu) → BN → MaxPool(2×2)      # 37×37×64
  ├─ Conv2D(128, 3×3, relu) → BN → MaxPool(2×2)     # 18×18×128
  ├─ Conv2D(256, 3×3, relu) → BN → MaxPool(2×2)     # 9×9×256
  │
  ├─ Flatten → Dense(512) → Dropout(0.5)
  ├─ Dense(256) → Dropout(0.3)
  └─ Dense(4, softmax)
```

In [ ]:

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.optimizers import Adam

train_g, val_g, test_g = make_generators(IMG_SIZE_CUSTOM)

custom_model = Sequential([
    Input(shape=(*IMG_SIZE_CUSTOM, 3)),
    Conv2D(32,  (3,3), activation='relu', padding='same'), BatchNormalization(), MaxPooling2D(2,2),
    Conv2D(64,  (3,3), activation='relu', padding='same'), BatchNormalization(), MaxPooling2D(2,2),
    Conv2D(128, (3,3), activation='relu', padding='same'), BatchNormalization(), MaxPooling2D(2,2),
    Conv2D(256, (3,3), activation='relu', padding='same'), BatchNormalization(), MaxPooling2D(2,2),
    Flatten(),
    Dense(512, activation='relu'), Dropout(0.5),
    Dense(256, activation='relu'), Dropout(0.3),
    Dense(NUM_CLASSES, activation='softmax')
], name='Custom_CNN')

custom_model.compile(optimizer=Adam(1e-3), loss='categorical_crossentropy', metrics=['accuracy'])
custom_model.summary()

t0 = time.time()
h_custom = custom_model.fit(
    train_g, epochs=EPOCHS_FROZEN + EPOCHS_FINETUNE,
    validation_data=val_g, callbacks=make_callbacks('custom'), verbose=1
)
custom_time = time.time() - t0
print(f"\nCustom CNN done in {custom_time/60:.1f} min.")

In [ ]:
import os, time, warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# Hyperparameters
BATCH_SIZE        = 32
EPOCHS_FROZEN     = 10
EPOCHS_FINETUNE   = 5
PATIENCE          = 5
IMG_SIZE_CUSTOM   = (150, 150)
IMG_SIZE_TRANSFER = (224, 224)

NUM_CLASSES = len([d for d in os.listdir(TRAIN_DIR) if os.path.isdir(os.path.join(TRAIN_DIR, d))])
CLASS_NAMES = sorted([d for d in os.listdir(TRAIN_DIR) if os.path.isdir(os.path.join(TRAIN_DIR, d))])

benchmark_results = []

print(f"Classes ({NUM_CLASSES}): {CLASS_NAMES}")
print(f"GPU available  : {len(tf.config.list_physical_devices('GPU')) > 0}")

# Callbacks factory
def make_callbacks(tag):
    return [
        EarlyStopping(monitor='val_accuracy', patience=PATIENCE,
                      restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_accuracy', factor=0.3, patience=3,
                          min_lr=1e-7, verbose=1),
        ModelCheckpoint(f'best_{tag}.keras', monitor='val_accuracy',
                        save_best_only=True, verbose=0),
    ]

# Generator factory
def make_generators(img_size, preprocess_fn=None, efficientnet=False):
    """Returns (train_gen, val_gen, test_gen) with correct preprocessing per model."""
    common_aug = dict(
        rotation_range=40,
        width_shift_range=0.2,
        height_shift_range=0.2,
        shear_range=0.2,
        zoom_range=0.2,
        horizontal_flip=True,
        brightness_range=[0.8, 1.2],
        fill_mode='nearest'
    )
    if preprocess_fn:
        # ResNet50 / VGG16: use their own preprocess_input (mean subtraction)
        tr_gen = ImageDataGenerator(preprocessing_function=preprocess_fn, **common_aug)
        vt_gen = ImageDataGenerator(preprocessing_function=preprocess_fn)
    elif efficientnet:
        # EfficientNetB0: NO rescale — the model has an internal rescaling layer
        # Applying rescale=1/255 here would double-scale pixels into [0, ~0.004]
        tr_gen = ImageDataGenerator(**common_aug)
        vt_gen = ImageDataGenerator()
    else:
        # Custom CNN: simple /255 normalisation to [0, 1]
        tr_gen = ImageDataGenerator(rescale=1./255, **common_aug)
        vt_gen = ImageDataGenerator(rescale=1./255)

    flow = dict(target_size=img_size, batch_size=BATCH_SIZE, class_mode='categorical')
    return (
        tr_gen.flow_from_directory(TRAIN_DIR, **flow),
        vt_gen.flow_from_directory(VAL_DIR,   **flow),
        vt_gen.flow_from_directory(TEST_DIR,  **flow, shuffle=False),
    )

# Evaluation helper
def evaluate_and_record(model, test_gen, name, h1, h2=None, t=0):
    test_gen.reset()
    loss, acc = model.evaluate(test_gen, verbose=0)
    test_gen.reset()
    preds  = model.predict(test_gen, verbose=0)
    y_pred = np.argmax(preds, axis=1)
    y_true = test_gen.classes

    print(f"\n{'='*55}\n  {name}\n{'='*55}")
    print(f"  Accuracy : {acc*100:.2f}%    Loss: {loss:.4f}    Time: {t/60:.1f} min")
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

    # Confusion matrix
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred),
                           display_labels=CLASS_NAMES).plot(ax=axes[0], cmap='Blues', colorbar=False)
    axes[0].set_title(f'{name} — Confusion Matrix')
    axes[0].tick_params(axis='x', rotation=30)

    # Training curves
    hs = [h for h in [h1, h2] if h]
    clrs = ['tab:blue', 'tab:orange']
    lbls = ['Phase 1 (frozen)', 'Phase 2 (fine-tune)']
    off  = 0
    for i, h in enumerate(hs):
        ep = range(off+1, off+len(h.history['accuracy'])+1)
        axes[1].plot(ep, h.history['val_accuracy'], color=clrs[i], lw=2, label=lbls[i])
        axes[1].plot(ep, h.history['accuracy'], color=clrs[i], lw=1.2, ls='--', alpha=0.5)
        off += len(h.history['accuracy'])
    axes[1].set_title(f'{name} — Val Accuracy'); axes[1].set_xlabel('Epoch')
    axes[1].legend(); axes[1].set_ylim(0, 1)
    plt.suptitle(name, fontsize=13, fontweight='bold'); plt.tight_layout(); plt.show()

    benchmark_results.append({
        'Model': name,
        'Test Accuracy (%)': round(acc*100, 2),
        'Test Loss': round(loss, 4),
        'Params (M)': round(model.count_params()/1e6, 2),
        'Train Time (min)': round(t/60, 1),
    })

print("Config, callbacks, generators, and evaluation helper ready.")

In [ ]:
evaluate_and_record(custom_model, test_g, 'Custom CNN', h_custom, t=custom_time)

---
## 8. Model 2 — EfficientNetB0

### Architecture

```
Input (224×224×3)  ← raw pixels [0,255], NO manual rescale
  │
  ├─ EfficientNetB0 backbone (ImageNet weights, frozen in Phase 1)
  │    Internal rescaling layer (built-in)
  │    MBConv blocks with Squeeze-and-Excitation (SE)
  │    Compound scaled: depth×1.0, width×1.0, resolution×224
  │
  ├─ GlobalAveragePooling2D              # 1280-dim feature vector
  ├─ Dense(512, relu) → Dropout(0.5)
  ├─ Dense(256, relu) → Dropout(0.3)
  └─ Dense(4, softmax)
```

In [ ]:
import tf_keras
from tf_keras.applications import EfficientNetB0
from tf_keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tf_keras.models import Model
from tf_keras.preprocessing.image import ImageDataGenerator
from tf_keras.optimizers import Adam
from tf_keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

# Callbacks using tf_keras
def make_eff_callbacks(tag):
    return [
        EarlyStopping(monitor='val_accuracy', patience=PATIENCE,
                      restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_accuracy', factor=0.3, patience=3,
                          min_lr=1e-7, verbose=1),
        ModelCheckpoint(f'best_{tag}.keras', monitor='val_accuracy',
                        save_best_only=True, verbose=0),
    ]

# Generators
eff_datagen_train = ImageDataGenerator(
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)
eff_datagen_val = ImageDataGenerator()

train_ge = eff_datagen_train.flow_from_directory(
    TRAIN_DIR, target_size=IMG_SIZE_TRANSFER, batch_size=BATCH_SIZE, class_mode='categorical'
)
val_ge = eff_datagen_val.flow_from_directory(
    VAL_DIR, target_size=IMG_SIZE_TRANSFER, batch_size=BATCH_SIZE, class_mode='categorical'
)
test_ge = eff_datagen_val.flow_from_directory(
    TEST_DIR, target_size=IMG_SIZE_TRANSFER, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=False
)

# Model
def build_efficientnet(num_classes):
    base = EfficientNetB0(
        input_shape=(*IMG_SIZE_TRANSFER, 3),
        include_top=False,
        weights='imagenet'
    )
    base.trainable = False

    inputs = tf_keras.Input(shape=(*IMG_SIZE_TRANSFER, 3))
    x = base(inputs, training=False)
    x = GlobalAveragePooling2D()(x)
    x = Dense(512, activation='relu')(x)
    x = Dropout(0.5)(x)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.3)(x)
    outputs = Dense(num_classes, activation='softmax')(x)
    return Model(inputs, outputs, name='EfficientNetB0'), base

eff_model, eff_base = build_efficientnet(NUM_CLASSES)
eff_model.compile(Adam(learning_rate=1e-3), 'categorical_crossentropy', ['accuracy'])
eff_model.summary()

# Phase 1: Frozen backbone
print("\n── Phase 1: Frozen backbone ──")
t0 = time.time()
h_eff1 = eff_model.fit(
    train_ge, epochs=EPOCHS_FROZEN, validation_data=val_ge,
    callbacks=make_eff_callbacks('eff_p1'), verbose=1
)

# Phase 2: Fine-tune last 30 layers
print("\n Phase 2: Fine-tune last 30 layers")
eff_base.trainable = True
for layer in eff_base.layers[:-30]:
    layer.trainable = False
eff_model.compile(Adam(learning_rate=1e-5), 'categorical_crossentropy', ['accuracy'])
h_eff2 = eff_model.fit(
    train_ge, epochs=EPOCHS_FINETUNE, validation_data=val_ge,
    callbacks=make_eff_callbacks('eff_p2'), verbose=1
)
eff_time = time.time() - t0
print(f"\nEfficientNetB0 done in {eff_time/60:.1f} min.")

In [ ]:
evaluate_and_record(eff_model, test_ge, 'EfficientNetB0', h_eff1, h_eff2, eff_time)

---
## 9. Model 3 — ResNet50

### Architecture

```
Input (224×224×3)  ← preprocess_input (BGR mean subtraction)
  │
  ├─ ResNet50 backbone (ImageNet weights, frozen in Phase 1)
  │    Conv1 stem
  │    Stage 2: 3 Residual blocks (64 filters)
  │    Stage 3: 4 Residual blocks (128 filters)
  │    Stage 4: 6 Residual blocks (256 filters)
  │    Stage 5: 3 Residual blocks (512 filters)  ← unfrozen in Phase 2
  │
  ├─ GlobalAveragePooling2D              # 2048-dim feature vector
  ├─ Dense(512, relu) → Dropout(0.5)
  ├─ Dense(256, relu) → Dropout(0.3)
  └─ Dense(4, softmax)
```

In [ ]:
from tf_keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
import tf_keras as keras
from tf_keras.applications import ResNet50
from tf_keras.applications.resnet50 import preprocess_input as resnet_pre
from tf_keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tf_keras.models import Model
from tf_keras.optimizers import Adam

train_gr, val_gr, test_gr = make_generators(IMG_SIZE_TRANSFER, preprocess_fn=resnet_pre)

def build_res(num_classes):
    base = ResNet50(input_shape=(*IMG_SIZE_TRANSFER,3), include_top=False, weights='imagenet')
    base.trainable = False
    inp = keras.Input(shape=(*IMG_SIZE_TRANSFER,3))
    x   = base(inp, training=False)
    x   = GlobalAveragePooling2D()(x)
    x   = Dense(512, activation='relu')(x); x = Dropout(0.5)(x)
    x   = Dense(256, activation='relu')(x); x = Dropout(0.3)(x)
    out = Dense(num_classes, activation='softmax')(x)
    return Model(inp, out, name='ResNet50'), base

res_model, res_base = build_res(NUM_CLASSES)
res_model.compile(Adam(1e-3), 'categorical_crossentropy', ['accuracy'])
res_model.summary()

print("\n Phase 1: Frozen backbone")
t0 = time.time()
h_res1 = res_model.fit(train_gr, epochs=EPOCHS_FROZEN,
                       steps_per_epoch=len(train_gr), validation_data=val_gr,
                       validation_steps=len(val_gr),
                       callbacks=make_callbacks('res_p1'), verbose=1)

print("\n Phase 2: Fine-tune conv5 block")
res_base.trainable = True
trainable = False
for layer in res_base.layers:
    if layer.name == 'conv5_block1_1_conv': trainable = True
    layer.trainable = trainable
res_model.compile(Adam(1e-5), 'categorical_crossentropy', ['accuracy'])
h_res2 = res_model.fit(train_gr, epochs=EPOCHS_FINETUNE,
                       steps_per_epoch=len(train_gr), validation_data=val_gr,
                       validation_steps=len(val_gr),
                       callbacks=make_callbacks('res_p2'), verbose=1)
res_time = time.time() - t0
print(f"\nResNet50 done in {res_time/60:.1f} min.")

In [ ]:
evaluate_and_record(res_model, test_gr, 'ResNet50', h_res1, h_res2, res_time)

---
## 10. Model 4 — VGG16

### Architecture

```
Input (224×224×3)  ← preprocess_input (BGR mean subtraction)
  │
  ├─ Block 1: Conv2D(64)×2  → MaxPool          # 112×112×64
  ├─ Block 2: Conv2D(128)×2 → MaxPool          # 56×56×128
  ├─ Block 3: Conv2D(256)×3 → MaxPool          # 28×28×256
  ├─ Block 4: Conv2D(512)×3 → MaxPool          # 14×14×512
  ├─ Block 5: Conv2D(512)×3 → MaxPool          # 7×7×512  ← unfrozen in Phase 2
  │
  ├─ GlobalAveragePooling2D  (replaces VGG's original Flatten+4096)
  ├─ Dense(512, relu) → Dropout(0.5)
  ├─ Dense(256, relu) → Dropout(0.3)
  └─ Dense(4, softmax)
```

In [ ]:
import tf_keras as keras
from tf_keras.applications import VGG16
from tf_keras.applications.vgg16 import preprocess_input as vgg_pre
from tf_keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tf_keras.models import Model
from tf_keras.optimizers import Adam

train_gv, val_gv, test_gv = make_generators(IMG_SIZE_TRANSFER, preprocess_fn=vgg_pre)

def build_vgg(num_classes):
    base = VGG16(input_shape=(*IMG_SIZE_TRANSFER,3), include_top=False, weights='imagenet')
    base.trainable = False
    inp = keras.Input(shape=(*IMG_SIZE_TRANSFER,3))
    x   = base(inp, training=False)
    x   = GlobalAveragePooling2D()(x)
    x   = Dense(512, activation='relu')(x); x = Dropout(0.5)(x)
    x   = Dense(256, activation='relu')(x); x = Dropout(0.3)(x)
    out = Dense(num_classes, activation='softmax')(x)
    return Model(inp, out, name='VGG16'), base

vgg_model, vgg_base = build_vgg(NUM_CLASSES)
vgg_model.compile(Adam(1e-3), 'categorical_crossentropy', ['accuracy'])
vgg_model.summary()

print("\n Phase 1: Frozen backbone")
t0 = time.time()
h_vgg1 = vgg_model.fit(train_gv, epochs=EPOCHS_FROZEN,
                       steps_per_epoch=len(train_gv), validation_data=val_gv,
                       validation_steps=len(val_gv),
                       callbacks=make_callbacks('vgg_p1'), verbose=1)

print("\n Phase 2: Fine-tune Block5")
vgg_base.trainable = True
for layer in vgg_base.layers:
    layer.trainable = layer.name.startswith('block5')
vgg_model.compile(Adam(1e-5), 'categorical_crossentropy', ['accuracy'])
h_vgg2 = vgg_model.fit(train_gv, epochs=EPOCHS_FINETUNE,
                       steps_per_epoch=len(train_gv), validation_data=val_gv,
                       validation_steps=len(val_gv),
                       callbacks=make_callbacks('vgg_p2'), verbose=1)
vgg_time = time.time() - t0
print(f"\nVGG16 done in {vgg_time/60:.1f} min.")

In [ ]:
evaluate_and_record(vgg_model, test_gv, 'VGG16', h_vgg1, h_vgg2, vgg_time)

---
## 11. Benchmark & Comparison

All four models are compared on the **held-out test set** (196 images never seen during training or validation).

The comparison covers:
- **Test accuracy** — primary metric
- **Test loss** — lower = more calibrated predictions
- **Total parameters** — proxy for inference speed and memory footprint
- **Training time** — practical deployment consideration

In [ ]:
import pandas as pd

df = (pd.DataFrame(benchmark_results)
        .sort_values('Test Accuracy (%)', ascending=False)
        .reset_index(drop=True))
df.index += 1

print("         BENCHMARK RESULTS — CROP DISEASE DETECTION")
print(df.to_string())


# Bar chart comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
models  = df['Model'].tolist()
colours = ['#43a047', '#1565c0', '#e65100', '#6a1b9a'][:len(models)]

for ax, col, title, fmt, ylim in [
    (axes[0], 'Test Accuracy (%)', 'Test Accuracy (%)', '{:.1f}%', (0,100)),
    (axes[1], 'Test Loss',         'Test Loss',          '{:.4f}',  None),
    (axes[2], 'Train Time (min)',   'Training Time (min)','{:.1f}',   None),
]:
    bars = ax.bar(models, df[col], color=colours, edgecolor='white', width=0.55)
    ax.set_title(title, fontweight='bold')
    if ylim: ax.set_ylim(*ylim)
    ax.spines[['top','right']].set_visible(False)
    ax.tick_params(axis='x', rotation=15)
    for bar in bars:
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.01,
                fmt.format(bar.get_height()), ha='center', va='bottom', fontsize=10)

plt.suptitle('Crop Disease Detection — Model Benchmark', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('benchmark.png', dpi=150, bbox_inches='tight')
plt.show()

winner = df.iloc[0]
print(f"\n Best model: {winner['Model']}  —  {winner['Test Accuracy (%)']:.2f}% test accuracy")

---
## 12. Interactive Inference
### Preprocessing at inference time

The same preprocessing contract used during training must be applied at inference. The `preprocess_image()` function routes each model to the correct transform — this is a common source of bugs when deploying models.

In [ ]:
# Model registry: name → (model, img_size, prep_mode)
MODEL_REGISTRY = {}
try: MODEL_REGISTRY['Custom CNN']     = (custom_model, IMG_SIZE_CUSTOM,   'rescale')
except NameError: pass
try: MODEL_REGISTRY['EfficientNetB0'] = (eff_model,    IMG_SIZE_TRANSFER, 'none')
except NameError: pass
try: MODEL_REGISTRY['ResNet50']       = (res_model,    IMG_SIZE_TRANSFER, 'resnet')
except NameError: pass
try: MODEL_REGISTRY['VGG16']          = (vgg_model,    IMG_SIZE_TRANSFER, 'vgg')
except NameError: pass

ACTIVE = df.iloc[0]['Model'] if benchmark_results else list(MODEL_REGISTRY)[0]
print(f"Active model : {ACTIVE}")
print(f"All models   : {list(MODEL_REGISTRY.keys())}")

In [ ]:
import json

ARTIFACT_DIR = 'artifacts/disease'
os.makedirs(ARTIFACT_DIR, exist_ok=True)

MODEL_FILENAMES = {
    'Custom CNN':     'custom.keras',
    'EfficientNetB0': 'efficientnet.keras',
    'ResNet50':       'resnet.keras',
    'VGG16':          'vgg16.keras',
}

exported = {}
for name, (mdl, img_size, mode) in MODEL_REGISTRY.items():
    fname = MODEL_FILENAMES.get(name, f"{name.lower().replace(' ', '_')}.keras")
    fpath = os.path.join(ARTIFACT_DIR, fname)
    mdl.save(fpath)
    exported[name] = {'file': fname, 'height': img_size[0], 'width': img_size[1], 'preprocess_mode': mode}
    print(f"Saved {name:<15} -> {fpath}")

class_names_map = {i: cls for i, cls in enumerate(CLASS_NAMES)}
with open(os.path.join(ARTIFACT_DIR, 'class_names.json'), 'w') as f:
    json.dump(class_names_map, f, indent=2)

# image_config.json — per-model preprocessing contract, keyed by the saved filename so the
# backend can look up "how do I prep an image for custom.keras" without guessing.
image_config = {
    info['file']: {'height': info['height'], 'width': info['width'], 'preprocess_mode': info['preprocess_mode']}
    for info in exported.values()
}
with open(os.path.join(ARTIFACT_DIR, 'image_config.json'), 'w') as f:
    json.dump(image_config, f, indent=2)

# metrics.json — full benchmark table plus which model the notebook recommends as default
with open(os.path.join(ARTIFACT_DIR, 'metrics.json'), 'w') as f:
    json.dump({'active_model': ACTIVE, 'results': benchmark_results}, f, indent=2)

print(f"\nArtifacts exported to '{ARTIFACT_DIR}/':")
for fn in sorted(os.listdir(ARTIFACT_DIR)):
    print(' -', fn)

# Optional: zip + download the artifact folder so you can commit it into the backend repo
try:
    from google.colab import files
    shutil.make_archive('disease_artifacts', 'zip', ARTIFACT_DIR)
    print("\nZipped to disease_artifacts.zip — downloading...")
    files.download('disease_artifacts.zip')
except ModuleNotFoundError:
    print("\nLocal Jupyter: artifacts are already on disk at", os.path.abspath(ARTIFACT_DIR))

In [ ]:
from PIL import Image
import io

HEALTHY_LABEL = 'Healthy'

def preprocess_image(pil_img, img_size, mode):
    """Resize + apply the correct preprocessing for each model."""
    img = np.array(pil_img.convert('RGB').resize(img_size), dtype=np.float32)
    if   mode == 'rescale': img = img / 255.0
    elif mode == 'resnet':
        from tensorflow.keras.applications.resnet50 import preprocess_input; img = preprocess_input(img)
    elif mode == 'vgg':
        from tensorflow.keras.applications.vgg16 import preprocess_input;    img = preprocess_input(img)
    # mode == 'none' (EfficientNet): pass raw pixels
    return np.expand_dims(img, 0)

def predict_leaf(pil_img, model_name=None):
    name = model_name or ACTIVE
    mdl, img_size, mode = MODEL_REGISTRY[name]
    probs = mdl.predict(preprocess_image(pil_img, img_size, mode), verbose=0)[0]
    idx, conf, label = int(np.argmax(probs)), float(np.max(probs)), CLASS_NAMES[int(np.argmax(probs))]

    healthy = (label == HEALTHY_LABEL)
    colour  = '#2e7d32' if healthy else '#c62828'
    status  = 'HEALTHY' if healthy else 'DISEASED'
    msg     = 'No disease detected. The leaf appears healthy.' if healthy else \
              f'Detected: {label}. Consider treatment and agronomist consultation.'

    fig, axes = plt.subplots(1, 2, figsize=(13,5), gridspec_kw={'width_ratios':[1,1.4]})
    fig.patch.set_facecolor('#f9f9f9')
    axes[0].imshow(pil_img); axes[0].axis('off')
    axes[0].set_title(f"{status}\n{name} | Confidence: {conf*100:.1f}%",
                      fontsize=13, fontweight='bold', color=colour)
    bar_c = ['#66bb6a' if c == label and healthy else
             '#ef5350' if c == label else '#ef9a9a' for c in CLASS_NAMES]
    bars = axes[1].barh(CLASS_NAMES, probs*100, color=bar_c, height=0.55)
    axes[1].set_xlim(0,100); axes[1].set_xlabel('Confidence (%)')
    axes[1].set_title('Class Probabilities'); axes[1].spines[['top','right','left']].set_visible(False)
    for bar, p in zip(bars, probs):
        axes[1].text(min(p*100+1.5,96), bar.get_y()+bar.get_height()/2,
                     f'{p*100:.1f}%', va='center', fontsize=10)
    plt.tight_layout(); plt.show()

    print(f'\n  {status}  |  {label}  |  {conf*100:.2f}%')
    print(f'  {msg}')
    if conf < 0.60:
        print('\n Low confidence — image may be out-of-distribution or ambiguous.')
    return label, conf

print('predict_leaf() ready. Run the next cell to upload an image.')

In [ ]:
# Upload & predict
try:
    from google.colab import files
    print('Upload a leaf image (JPG/PNG):')
    uploaded = files.upload()
    for fname, data in uploaded.items():
        img = Image.open(io.BytesIO(data))
        print(f'\nFile: {fname}  ({img.size[0]}×{img.size[1]})')
        predict_leaf(img)
except ModuleNotFoundError:
    import ipywidgets as widgets; from IPython.display import display
    uploader = widgets.FileUpload(accept='image/*', multiple=False)
    out = widgets.Output()
    def _on_upload(change):
        with out:
            out.clear_output()
            for n, m in uploader.value.items():
                predict_leaf(Image.open(io.BytesIO(m['content'])))
    uploader.observe(_on_upload, names='value')
    display(uploader, out)

In [ ]:
# Compare ALL models on the same image
try:
    from google.colab import files
    print(' Upload ONE image to compare all models:')
    uploaded = files.upload()
    for fname, data in uploaded.items():
        img = Image.open(io.BytesIO(data)).convert('RGB')
        print(f'\nImage: {fname}\n')
        print(f'{"Model":<20} {"Prediction":<22} {"Confidence":>12}  Verdict')
        for mname, (mdl, isz, mode) in MODEL_REGISTRY.items():
            p    = mdl.predict(preprocess_image(img, isz, mode), verbose=0)[0]
            lbl  = CLASS_NAMES[np.argmax(p)]
            conf = float(np.max(p))
            verdict = 'Healthy' if lbl == HEALTHY_LABEL else f'{lbl}'
            print(f'{mname:<20} {lbl:<22} {conf*100:>10.1f}%  {verdict}')
        print('─'*70)
except ModuleNotFoundError:
    print('Local Jupyter: pass PIL image to predict_leaf(img, model_name=<name>)')